# JAX-accelerated Galactic prior

This notebook demonstrates the JAX backend for the Galactic prior:

- `JaxHistogramDensity` — wraps the same histogram files as `HistogramDensity`
  but exposes a `log_density` method that is JIT-compilable
- `JaxGalacticPrior` — identical interface to `GalacticPrior` but built on
  JAX operations throughout

Key use-cases covered below:

1. Converting a NumPy density to the JAX backend
2. `jax.jit` compilation and timing comparison
3. `jax.vmap` for evaluating a batch of parameter vectors at once
4. `jax.grad` — limitations due to nearest-block lookup
5. Attaching a `BinaryCircularParameterization` to `JaxGalacticPrior`

In [ ]:
from pathlib import Path
import sys
import time

cwd = Path.cwd().resolve()
if (cwd / "src" / "gapmoe").exists():
    repo_root = cwd
elif (cwd.parent / "src" / "gapmoe").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError("Run this notebook from the repository root or example directory.")

sys.path.insert(0, str(repo_root / "src"))

import jax
import jax.numpy as jnp
import numpy as np

from gapmoe import HistogramDensity, JaxHistogramDensity, JaxGalacticPrior, PreRunner
from gapmoe.parameterizations import (
    BinaryCircularParameterization,
    calc_vEarth,
)

repo_root

## Load the NumPy density, then convert to JAX

`JaxHistogramDensity.from_numpy` copies the histogram arrays from a
`HistogramDensity` instance into JAX device arrays.  No re-reading of the
underlying files is needed.

In [ ]:
genulens_root = (repo_root / ".." / "genulens").resolve()
if not (genulens_root / "pre_gapmoe").exists():
    raise FileNotFoundError(
        f"Genulens pre_gapmoe was not found at {genulens_root / 'pre_gapmoe'}. "
        "Set genulens_root to your local Genulens checkout."
    )

runner = PreRunner(
    genulens_root=genulens_root,
    output_dir=repo_root / "example" / "pre_runner_outputs",
)

pre_run = runner.run(
    ra_deg=270.0,
    dec_deg=-30.0,
    run_name="emcee_demo",
    seed=123,
)

numpy_density = HistogramDensity.from_pre_run(pre_run)
jax_density = JaxHistogramDensity.from_numpy(numpy_density)

print("NumPy density type:", type(numpy_density).__name__)
print("JAX density type:  ", type(jax_density).__name__)

## Build `JaxGalacticPrior` and verify outputs match

In [ ]:
from gapmoe import GalacticPrior

np_prior = GalacticPrior(numpy_density)
jax_prior = JaxGalacticPrior(jax_density)

# A single physical-parameter tuple
ML, DL, DS, mu_N, mu_E = 0.3, 5.0, 8.0, 5.0, 2.0

lp_np = np_prior.log_prob(ML, DL, DS, mu_N, mu_E)
lp_jax = jax_prior.log_prob(ML, DL, DS, mu_N, mu_E)

print(f"NumPy log_prob:  {lp_np:.6f}")
print(f"JAX   log_prob:  {float(lp_jax):.6f}")

## JIT compilation

Wrapping `jax_prior.log_prob` with `jax.jit` fuses all JAX operations into a
single compiled kernel.  The first call pays the compilation cost; subsequent
calls are significantly faster.

In [ ]:
jit_log_prob = jax.jit(jax_prior.log_prob)

# Warm-up (compilation)
_ = jit_log_prob(ML, DL, DS, mu_N, mu_E).block_until_ready()

N = 500

t0 = time.perf_counter()
for _ in range(N):
    np_prior.log_prob(ML, DL, DS, mu_N, mu_E)
t_np = (time.perf_counter() - t0) / N * 1e6

t0 = time.perf_counter()
for _ in range(N):
    jit_log_prob(ML, DL, DS, mu_N, mu_E).block_until_ready()
t_jit = (time.perf_counter() - t0) / N * 1e6

print(f"NumPy  per call: {t_np:.1f} µs")
print(f"JAX/JIT per call: {t_jit:.1f} µs")
print(f"Speedup: {t_np / t_jit:.1f}x")

## Batch evaluation with `jax.vmap`

`jax.vmap` vectorises `jit_log_prob` over a batch of parameter tuples,
evaluating all at once without a Python loop.

In [ ]:
def log_prob_single(params):
    """Accepts a length-5 array [ML, DL, DS, mu_N, mu_E]."""
    return jax_prior.log_prob(params[0], params[1], params[2], params[3], params[4])


batch_log_prob = jax.jit(jax.vmap(log_prob_single))

key = jax.random.PRNGKey(0)
batch = jax.random.uniform(key, shape=(1024, 5)) * jnp.array([1.0, 10.0, 12.0, 20.0, 20.0])
batch = batch.at[:, 0].set(jnp.clip(batch[:, 0], 0.08, 1.5))   # ML
batch = batch.at[:, 1].set(jnp.clip(batch[:, 1], 0.1, 10.0))   # DL
batch = batch.at[:, 2].set(jnp.clip(batch[:, 2], 0.5, 12.0))   # DS

# Warm-up
_ = batch_log_prob(batch).block_until_ready()

t0 = time.perf_counter()
lp_batch = batch_log_prob(batch).block_until_ready()
t_vmap = (time.perf_counter() - t0) * 1e3

print(f"Batch of {len(batch):,} evaluations: {t_vmap:.1f} ms")
print(f"Finite fraction: {jnp.isfinite(lp_batch).mean():.2%}")

## Gradients with `jax.grad`

`jax.grad` can differentiate through `jax_prior.log_prob`, but the result
should be interpreted with care.

The murel histogram uses a **nearest-block lookup**: each `(DL, DS)` pair is
mapped to the nearest pre-computed histogram cell, and the density is then
looked up by linear interpolation in `mu`.  This means that the Jacobian with
respect to DL and DS is **exactly zero almost everywhere** — the function is
piecewise constant in those directions.

Gradients with respect to `mu_N` and `mu_E` are meaningful because they pass
through the linear interpolation step.

In [ ]:
grad_log_prob = jax.jit(jax.grad(log_prob_single))

point = jnp.array([ML, DL, DS, mu_N, mu_E])
g = grad_log_prob(point)

for name, val in zip(["ML", "DL", "DS", "mu_N", "mu_E"], g):
    print(f"d(log_prob)/d({name}) = {float(val):.6f}")

print()
print("Note: DL and DS gradients are exactly 0 due to nearest-block murel lookup.")
print("mu_N and mu_E gradients pass through the linear interpolation and are meaningful.")

## Using `JaxGalacticPrior` with a parameterization

`JaxGalacticPrior` accepts the same `Parameterization` protocol as
`GalacticPrior`.  When a parameterization is provided, `log_prob` takes the
full light-curve parameter vector and a context dict.

In [ ]:
t0_jd = 2460000.0
v_N, v_E = calc_vEarth(t0_jd, ra_deg=270.0, dec_deg=-30.0)
ctx = {"thS": 0.5, "vEarth": (v_N, v_E)}

param = BinaryCircularParameterization()
jax_prior_param = JaxGalacticPrior(jax_density, parameterization=param)

theta = jnp.array([
    t0_jd,   # t0
    50.0,    # tE
    0.1,     # u0
    0.005,   # rho
    0.1,     # q
    1.0,     # s
    0.5,     # alpha
    0.1,     # piEN
    0.05,    # piEE
    0.001,   # gamma1
    0.002,   # gamma2
    -0.003,  # gamma3
])

lp = jax_prior_param.log_prob(theta, context=ctx)
print(f"log prior (light-curve space, JAX) = {float(lp):.4f}")

### JIT-compile the full light-curve-space prior

The parameterization's JAX kernels (`_lc_to_phys_circular`, `_jacobian_circular`)
are already decorated with `@jax.jit`, so wrapping the outer `log_prob` with
another `jit` compiles everything together.

In [ ]:
def lp_from_theta(theta):
    return jax_prior_param.log_prob(theta, context=ctx)


jit_lp = jax.jit(lp_from_theta)

# Warm-up
_ = jit_lp(theta).block_until_ready()

t0 = time.perf_counter()
for _ in range(200):
    jit_lp(theta).block_until_ready()
t_jit = (time.perf_counter() - t0) / 200 * 1e6

print(f"JIT log_prob (12-dim light-curve space): {t_jit:.1f} µs per call")
print(f"log prior = {float(jit_lp(theta)):.4f}")